# Week 3 · Differentiable Logic Layers

**Repository:** `bitwise-llm-forge` · **Theory doc:** [`docs/theory/week3_difflogic.md`](../docs/theory/week3_difflogic.md)

---

### Learning objectives

1. Enumerate the 16 binary boolean gates and express each as a smooth polynomial in $(a,b)\in[0,1]^2$.
2. Formulate **gate selection** as a categorical distribution over logits $\alpha\in\mathbb{R}^{16}$ — making the choice itself differentiable.
3. Build a `DiffLogicLayer` with **fixed random wiring** and per-neuron learnable gate logits.
4. Train it on a binarised MNIST classification head and compare to a parameter-matched MLP baseline.
5. Visualize which gates the network ends up selecting — interpretability "for free".


In [ ]:
# Make ``src/`` importable when this notebook is launched from anywhere.
from pathlib import Path
import sys

_here = Path.cwd()
for candidate in (_here, *_here.parents):
    if (candidate / "src").is_dir():
        sys.path.insert(0, str(candidate))
        break

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

from src.utils.seeding import set_seed
set_seed(42)

print("environment ready · torch", __import__("torch").__version__)


## 1 · Mathematical derivation

### 1.1 The sixteen gates as polynomials

For $a,b\in[0,1]$ each binary boolean gate has a unique multilinear extension:

| Gate | Formula |
|---|---|
| AND | $a\,b$ |
| OR | $a + b - a\,b$ |
| XOR | $a + b - 2 a b$ |
| NAND | $1 - a\,b$ |
| NOT A | $1 - a$ |
| A | $a$ |
| FALSE / TRUE | $0$ / $1$ |
| ... | (16 total — see `src/difflogic/gates.py`) |

Each is a polynomial of degree ≤ 2, hence smooth and differentiable everywhere.

### 1.2 Soft gate selection

Each neuron stores a learnable logit vector $\alpha\in\mathbb{R}^{16}$. Its output is the **softmax-weighted mixture** of the 16 gate outputs on its two assigned inputs $(x_{a_j}, x_{b_j})$:

$$
g_\alpha(a,b) = \sum_{k=1}^{16}\operatorname{softmax}(\alpha)_k\,g_k(a,b).
$$

During training $\alpha$ is updated by ordinary backprop. At inference each neuron collapses to a single hard gate:

$$
k^* = \arg\max_k \alpha_k, \qquad y = g_{k^*}(a,b).
$$

After this discretization, the entire forward pass is **integer-only** — no floating-point multiplies anywhere.

### 1.3 Fixed random wiring

The pair of input indices $(a_j, b_j)$ for each output neuron is **drawn once at initialization** and never updated. This mirrors the sparse, irregular fan-in of physical combinational logic and keeps the per-layer parameter count at $\Theta(L\,n)$ (only the gate logits are learnable).


## 2 · Reference implementation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from src.difflogic import DiffLogicLayer, GATE_FUNCTIONS, NUM_GATES, apply_all_gates

# Visualise all 16 gates on the [0,1]² grid
grid = torch.linspace(0, 1, 41)
A, B = torch.meshgrid(grid, grid, indexing="xy")
fig, axes = plt.subplots(4, 4, figsize=(11, 11))
GATE_NAMES = ["FALSE","AND","A∧¬B","A","¬A∧B","B","XOR","OR",
              "NOR","XNOR","¬B","A∨¬B","¬A","¬A∨B","NAND","TRUE"]
for k, g in enumerate(GATE_FUNCTIONS):
    ax = axes.flat[k]
    img = g(A, B).numpy()
    ax.imshow(img, origin="lower", extent=[0,1,0,1], vmin=0, vmax=1, cmap="viridis")
    ax.set_title(f"{k}: {GATE_NAMES[k]}", fontsize=9)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
plt.suptitle("The 16 binary boolean gates as multilinear polynomials", y=1.00)
plt.tight_layout(); plt.show()


## 3 · Sanity checks

In [ ]:
torch.manual_seed(0)

# 3.1 Shape contract
layer = DiffLogicLayer(in_features=20, out_features=10)
x = torch.rand(8, 20)
y = layer(x)
print("input :", tuple(x.shape), "  output:", tuple(y.shape))
assert y.shape == (8, 10)

# 3.2 Output is in [0, 1] for inputs in [0, 1]
print("min/max output values:", float(y.min().item()), "/", float(y.max().item()))
assert (y >= 0).all() and (y <= 1).all()

# 3.3 Gradient flow
y.sum().backward()
print("gate_logits.grad max abs:", float(layer.gate_logits.grad.abs().max().item()))
assert layer.gate_logits.grad.abs().max() > 0

# 3.4 Discretization returns a valid gate index per neuron
idx = layer.discretize()
print("discrete gate indices:", idx.tolist())
assert idx.min() >= 0 and idx.max() < NUM_GATES


## 4 · Training on a tiny MNIST head

A real DiffLogic model needs several deep layers; for a notebook benchmark we use a *single* DiffLogic layer as a logit head on top of a fixed linear pixel embedding.
We compare it to a parameter-matched FP MLP.

To keep the notebook self-contained we use a small synthetic surrogate — `make_classification` shapes are similar to MNIST and the experiment runs in seconds.


In [ ]:
# Pure-torch synthetic classification — 10 classes in 64-dim input space,
# inputs squashed to [0, 1] (as DiffLogic gates expect).
def make_synthetic_classification(n: int = 4000, d: int = 64,
                                   n_classes: int = 10, seed: int = 0):
    g = torch.Generator().manual_seed(seed)
    centers = torch.randn(n_classes, d, generator=g) * 1.5
    y = torch.randint(0, n_classes, (n,), generator=g)
    x = centers[y] + 0.6 * torch.randn(n, d, generator=g)
    # Sigmoid -> [0, 1]
    x = torch.sigmoid(x)
    return x, y


X, Y = make_synthetic_classification()
split = int(0.75 * len(X))
X_tr, X_va = X[:split], X[split:]
y_tr, y_va = Y[:split], Y[split:]
print("train:", tuple(X_tr.shape), "  val:", tuple(X_va.shape))
print("class distribution (train):",
      torch.bincount(y_tr, minlength=10).tolist())


In [ ]:
# Model A: DiffLogic layer with 256 logic gates → 10-class linear head
class DiffLogicHead(nn.Module):
    def __init__(self, in_dim: int, hidden: int = 256, n_classes: int = 10):
        super().__init__()
        self.logic = DiffLogicLayer(in_dim, hidden)
        self.cls   = nn.Linear(hidden, n_classes)
    def forward(self, x):
        return self.cls(self.logic(x))


# Model B: parameter-matched MLP baseline
#   DiffLogic params ≈ hidden * 16 + (hidden + 1) * n_classes
#                    = 256*16 + 257*10 = 6666 (most in cls head)
#   MLP params       = in_dim*h + h + h*n_classes + n_classes = 64h + h + 10h + 10 = 75h + 10
#   Solve 75h + 10 ≈ 6666 → h ≈ 88
class MLPBaseline(nn.Module):
    def __init__(self, in_dim: int, hidden: int = 88, n_classes: int = 10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, n_classes),
        )
    def forward(self, x): return self.net(x)


def count(m): return sum(p.numel() for p in m.parameters())
m_logic = DiffLogicHead(64)
m_mlp   = MLPBaseline(64)
print(f"DiffLogic params : {count(m_logic):>6d}")
print(f"MLP        params: {count(m_mlp):>6d}")


In [ ]:
def train(model: nn.Module, epochs: int = 30, lr: float = 5e-3) -> tuple[list[float], list[float]]:
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    tr_losses, va_accs = [], []
    for ep in range(epochs):
        # full-batch (data is small)
        opt.zero_grad(set_to_none=True)
        loss = F.cross_entropy(model(X_tr), y_tr)
        loss.backward(); opt.step()
        with torch.no_grad():
            preds = model(X_va).argmax(-1)
            acc = float((preds == y_va).float().mean().item())
        tr_losses.append(float(loss.item())); va_accs.append(acc)
    return tr_losses, va_accs


from src.utils.seeding import set_seed
set_seed(0); tr_l, va_a = train(DiffLogicHead(64))
set_seed(0); tr_m, va_m = train(MLPBaseline(64))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(tr_l, label="DiffLogic"); axes[0].plot(tr_m, label="MLP")
axes[0].set_title("Training loss"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(va_a, label="DiffLogic"); axes[1].plot(va_m, label="MLP")
axes[1].set_title("Validation accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f"\nfinal val acc — DiffLogic: {va_a[-1]:.3f}  |  MLP: {va_m[-1]:.3f}")


## 5 · Which gates does the network end up using?

In [ ]:
# Take the trained DiffLogic model and look at the discrete gate distribution.
set_seed(0); model = DiffLogicHead(64); train(model)
gate_idx = model.logic.discretize().tolist()

import collections
counts = collections.Counter(gate_idx)
order = sorted(range(NUM_GATES), key=lambda k: -counts[k])
freqs = [counts[k] for k in order]

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.bar(range(NUM_GATES), freqs, color="#1f77b4")
ax.set_xticks(range(NUM_GATES))
ax.set_xticklabels([GATE_NAMES[k] for k in order], rotation=45, ha="right")
ax.set_ylabel("number of neurons"); ax.set_title("Gate-type distribution after training")
plt.tight_layout(); plt.show()

print("top 5 gates used:")
for k in order[:5]:
    print(f"  {GATE_NAMES[k]:8s}  {counts[k]:3d} neurons")


## 6 · Discussion

- **Collapse to a few gates** is the typical empirical finding — most neurons converge to AND / NAND / NOT-A / NOT-B, the ones that carry useful information bits. This is interpretable in a way an MLP weight matrix isn't.
- **Single layer is too shallow.** Petersen et al. (2022) stack 4–8 DiffLogic layers and report parity with parameter-matched MLPs on MNIST / CIFAR-10. With deep stacks plus the sigmoid-annealing schedule discussed in `docs/theory/week3_difflogic.md`, accuracy gaps close.
- **Hardware story.** After discretization the network is a sparse combinational circuit — implementable on FPGAs / ASICs at orders-of-magnitude lower energy than floating-point MLPs.
- **Why this lives in an LLM repo.** DiffLogic is an alternative axis of low-precision: instead of compressing the *weights*, it compresses the *operator*. It's a useful counterpoint to BitNet's weight ternarization and points at hybrid futures.

### References
- Petersen, F. et al. "Deep Differentiable Logic Gate Networks." NeurIPS 2022.
- Bishop, C. M. & Bishop, H. *Deep Learning: Foundations and Concepts*, Springer 2024 (ch. 6 on representational capacity).
